In [15]:
import os
import glob
import tiktoken
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [7]:
knowledge = {}

filenames = glob.glob("knowledge/profile/*")

for filename in filenames:
    name = Path(filename).stem.split("-")[-1].lower()
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [9]:
knowledge["interests"]

'---\nid: "profile-perspective"\ntitle: "Favour\'s engineering perspective and interests"\ntopic: "profile"\ntopics:\n  - "Engineering philosophy"\n  - "Frontend engineering"\n  - "Interface performance"\n  - "Security"\n  - "AI safety"\n  - "Agentic AI"\n  - "Large language models"\n  - "Video games"\nentities: ["Favour Chukwuemerie Ututu"]\nsource: "rag/sources/original-portfolio-knowledge.md"\nsource_sections: ["In Favour\'s words > How I think about engineering", "In Favour\'s words > What I enjoy"]\n---\n\n# Favour\'s engineering perspective and interests\n\nThe following passages are Favour Chukwuemerie Ututu\'s own words.\n\n## Favour\'s engineering perspective — in his own words\n\nI started my career as a frontend engineer, so I\'ve always loved seeing beautiful user interfaces. But they have to work, too. I love an interface that looks good, feels easy to use, and loads quickly. I don\'t want to wait a minute for a page to load just to do something simple.\n\nAs I kept buildi

In [10]:
knowledge.keys()

dict_keys(['overview', 'interests'])

In [11]:
# How many characters in all the documents?

knowledge_base_path = "knowledge/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 12 files in the knowledge base
Total characters in knowledge base: 26,111


In [13]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-mini: 5,394


In [16]:

knowledge_dir = Path("knowledge")
documents = []

for folder in sorted(knowledge_dir.iterdir()):
    if not folder.is_dir():
        continue

    doc_type = folder.name

    for filepath in sorted(folder.rglob("*.md")):
        doc = Document(
            page_content=filepath.read_text(encoding="utf-8"),
            metadata={
                "source": filepath.as_posix(),
                "doc_type": doc_type,
            },
        )
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 12 documents


In [22]:
print(documents[1])

page_content='---
id: "experience-flamong-policy-ng"
title: "Favour at Flamong — Policy.ng frontend engineering"
topic: "experience"
topics:
  - "Frontend engineering"
  - "API integration"
  - "Form validation"
  - "Authentication and permissions"
  - "Content administration"
  - "Software testing"
  - "AI-assisted interfaces"
entities: ["Favour Chukwuemerie Ututu", "Flamong", "Policy.ng"]
source: "rag/sources/original-portfolio-knowledge.md"
source_sections: ["Professional experience > Flamong — Frontend Developer Intern", "What Favour works on > Frontend engineering", "About Favour"]
additional_sources:
  - source: "rag/sources/confirmed-portfolio-details.md"
    source_sections: ["Current role and start date"]
---

# Favour at Flamong — Policy.ng frontend engineering

**Position:** Frontend Developer Intern at Flamong.  
**Employment period:** May 2026–Present, confirmed on 10 November 2026.  
**Product:** [Policy.ng](https://policy.ng/)  
**Role:** Frontend engineering for the pub